# **Opis Zadania Koncowego**

# 1. Określenie problemu biznesowego**

Celem modelu jest przewidzenie, czy klient nie spłaci zobowiązania w następnym miesiącu:
    
0 – klient spłaci zobowiązanie,
1 – klient nie spłaci zobowiązania.
    
Na tej podstawie bank może podjąć decyzję:
- niskie prawdopodobieństwo braku spłaty → można rozważyć przyznanie karty,
- wysokie prawdopodobieństwo → odmowa albo dodatkowa weryfikacja.
    
Model nie powinien samodzielnie podejmować ostatecznej decyzji. Jest narzędziem wspierającym analityka kredytowego.

# 2. Importowanie bibliotek

Zaimportowano biblioteki potrzebne do:
    
- przetwarzania danych: pandas, numpy,
- wizualizacji: matplotlib, seaborn,
- budowania modeli: scikit-learn,
- zapisywania modelu: joblib.
    
Ustawiono również:
RANDOM_STATE = 2022 -Dzięki temu losowy podział danych i trenowanie modeli dają powtarzalne wyniki.

# 3. Wczytanie danych

Oryginalny plik .xls zawiera dwa wiersze nagłówkowe. Właściwe nazwy kolumn znajdują się w drugim wierszu, dlatego użyto:

-pd.read_excel(xls_path, header=1)

Ponieważ odczyt starych plików .xls wymaga biblioteki xlrd, przygotowany został również odpowiednik CSV. Notebook:
- próbuje odczytać oryginalny plik .xls,
-j eżeli nie jest to możliwe, używa pliku CSV.
    
Po wczytaniu otrzymaliśmy:
30 000 obserwacji
25 kolumn
Wśród nich znajduje się:
ID,
23 zmienne opisujące klienta,
zmienna docelowa default payment next month.

# 4. Sprawdzenie jakości danych

Sprawdzono:
- typy kolumn,
- brakujące wartości,
- liczbę unikalnych wartości,
- duplikaty identyfikatora,
- powtarzające się profile klientów,
- zakresy i kody poszczególnych zmiennych.
    
Nie znaleziono wartości NaN ani powtórzonych identyfikatorów.
    
Znaleziono jednak błędne lub nieudokumentowane kategorie.

EDUCATION
Według opisu prawidłowe kody to:

1 – szkoła wyższa
2 – uniwersytet
3 – szkoła średnia
4 – inne

W danych występowały również:
0, 5, 6
Połączono je z kategorią 4, czyli „inne”:
dataframe['EDUCATION'] = dataframe['EDUCATION'].replace({
    0: 4,
    5: 4,
    6: 4
})


Zrobiono to, ponieważ nie znamy ich dokładnego znaczenia, ale wiadomo, że nie należą do trzech podstawowych kategorii.
MARRIAGE
Według opisu prawidłowe kody to:
1 – małżeństwo
2 – osoba stanu wolnego
3 – inne
W danych pojawił się również kod 0. Został połączony z kategorią 3:
dataframe['MARRIAGE'] = dataframe['MARRIAGE'].replace({0: 3})

Statusy spłaty
W kolumnach PAY_* występują wartości -2, -1, 0, a następnie dodatnie wartości opóźnień.
Nie usunięto -2 i 0, ponieważ są to rzeczywiste kody używane w oryginalnym zbiorze. Mogą opisywać brak użycia kredytu lub kredyt odnawialny.

Duplikaty

Znaleziono 35 identycznych profili po usunięciu ID. Nie zostały automatycznie usunięte, ponieważ mogą reprezentować różnych klientów o takich samych, zaokrąglonych wartościach.
    

# 5. Analiza zmiennej docelowej

Sprawdzono rozklad klas:

- spłata, klasa 0:     23 364 klientów - 77,88%
- brak spłaty klasa 1: 6 636 klientów - 22,12%

Oznacza to, że dane są niezbalansowane!
Model który zawsze przewiduje klasę 0, osiągnąłby 77,12 % procent accuracy.

Jednocześnie nie wykryłby ani jednego klienta, który nie spłaci zobowiązania. Dlatego samo accuracy nie wystarcza do oceny modelu.
Zastosowano dodatkowe miary:
- recall – jaki procent klientów niespłacających wykryto,
- precision – jaki procent klientów oznaczonych jako ryzykowni rzeczywiście nie spłacił,
- F1 – kompromis pomiędzy precision i recall,
- F2 – mocniej premiuje wykrywanie ryzykownych klientów,
- ROC-AUC – zdolność modelu do rozróżniania klas,
- PR-AUC – jakość wykrywania klasy pozytywnej w niezbalansowanych danych,
- balanced accuracy – średnia skuteczność dla obu klas.

# 6. Analiza eksploracyjna

Sprawdzono zależności pomiędzy brakiem spłaty a:
- limitem kredytowym,
- wiekiem,
- historią opóźnień,
- płcią,
- wykształceniem,
- stanem cywilnym.
    
Szczególnie ważne okazały się statusy spłaty. Im większe i częstsze opóźnienia, tym wyższy udział klientów, którzy nie spłacili zobowiązania w kolejnym miesiącu.
Analiza zmiennych takich jak płeć czy stan cywilny powinna być traktowana ostrożnie. Przed wdrożeniem należy sprawdzić przepisy oraz ryzyko dyskryminacji klientów.

# 7. Inżynieria cech

Na podstawie istniejących danych utworzono dodatkowe zmienne.
    
Średni rachunek
BILL_MEAN_6M
Jest to średnia wartość rachunków klienta z sześciu miesięcy.
    
Średnia spłata
PAYMENT_MEAN_6M
Pokazuje przeciętną kwotę wpłacaną przez klienta.
    
Wykorzystanie limitu
UTILIZATION_MEAN

Obliczono jako:
średni rachunek / limit kredytowy
    
Przykład:
średni rachunek = 80 000
limit = 100 000
wykorzystanie = 0,8
Klient wykorzystuje więc średnio 80% dostępnego limitu.
    
Stosunek spłat do rachunków
PAYMENT_TO_BILL
Porównuje sumę wpłat z sumą dodatnich rachunków. Niska wartość może wskazywać, że klient spłaca niewielką część zadłużenia.
    
Liczba miesięcy z opóźnieniem
DELAY_MONTHS_COUNT
Liczy, w ilu z sześciu miesięcy klient miał dodatni status opóźnienia.
    
Największe opóźnienie
MAX_DELAY
Pokazuje największe zarejestrowane opóźnienie klienta.
    
Zmiana rachunku i spłaty
RECENT_BILL_CHANGE
RECENT_PAYMENT_CHANGE
Pokazują, jak zmieniły się zadłużenie oraz kwoty wpłat pomiędzy kwietniem i wrześniem.
    
Te cechy zostały utworzone, aby model otrzymał gotowe informacje podsumowujące zachowanie klienta.


# 8. Przygotowanie X i y

Zmienna docelowa została zapisana jako:

y = dataframe['default payment next month']

Cechy modelu zapisano jako:
X = dataframe.drop(columns=[
    'default payment next month',
    'ID'
])

Usunięto ID, ponieważ jest tylko identyfikatorem. Nie opisuje zdolności klienta do spłaty zobowiązania.

# 9. Podział danych

Dane podzielono na trzy części:

- zbiór treningowy:   19 200 obserwacji
- zbiór walidacyjny:   4 800 obserwacji
- zbiór testowy:       6 000 obserwacji

Zastosowano:
stratify=y

Dzięki temu każdy zbiór zawiera około 22,12% przypadków braku spłaty.
    
Zbiory pełnią różne funkcje:
- treningowy – służy do uczenia modeli,
- walidacyjny – służy do wyboru modelu i progu,
- testowy – służy wyłącznie do końcowej oceny.
    
Nie dobierano parametrów ani progu na zbiorze testowym, ponieważ prowadziłoby to do wycieku informacji.

# 10. Model bazowy

DummyClassifier(strategy='prior')

Model bazowy nie analizuje klientów. Korzysta jedynie z udziału klas.

Wyniki:
- accuracy:  0,7788
- recall:    0
- F1:        0
- F2:        0
- ROC-AUC:   0,5
- PR-AUC:    0,2213

Wysokie accuracy jest pozorne. Model nie wykrył żadnego klienta niespłacającego zobowiązania.
Modele właściwe powinny osiągnąć wyniki lepsze od tego punktu odniesienia.

# 11.Trenowanie MOdeli

Wybrano 3 dla porównania.
    
11.1 Regresja logistyczna 
    LogisticRegression
    Przed modelem użyto StandardScaler, ponieważ regresja logistyczna korzysta na porównywalnej skali cech.
    Zastosowano 'class_weight='balanced' dla nadania większego znaczenia rzadszej klasie 1.

11.2 Drzewo decyzyjne
    DecisionTreeClassifier
    Model tworzy reguły w rodzaju:
    'jeżeli PAY_0 > 1 i MAX_DELAY > 2, to ryzyko jest wysokie'
    Drzewo jest łatwe do interpretacji, ale może nadmiernie dopasować się do danych.

11.3 Random Forest
    RandomForestClassifier
    RAndom Forest buduje wiele drzew i łączy ich wyniki. Zwykle jest stabilniejszy od pojedynczego drzewa.

# 12. Strojenie hiperparametrów

Zastosowano GridSearchCV.
    
Dla każdego modelu sprawdzono różne ustawienia. 
Walidacja została wykonana za pomocą:
 - StratifiedKFold(n_splits=4) 
    Każdy model był oceniany na czterech podziałach danych, zachowujących udział obu klas.
     
Główną miarą podczas strojenia było:
        - scoring='average_precision'
    czyli PR-AUC. Jest ona odpowiednia dla niezbalansowanych danych.

Najlepsze parametry Random Forest:
    {
    'max_depth': 10,
    'min_samples_leaf': 5
}

Oznacza to:
- każde drzewo mogło mieć maksymalnie 10 poziomów,
- każdy końcowy liść musiał zawierać minimum 5 obserwacji.


# 13. Dobór progu decyzyjnego

Standardowo klasyfikator stosuje próg: '0.5'

Czyli:
- prawdopodobieństwo ≥ 0,5 → przewidywany brak spłaty
- prawdopodobieństwo < 0,5 → przewidywana spłata

W projekcie próg został dobrany na zbiorze walidacyjnym tak, aby maksymalizować F2. i wynosi '0.245'

Oznacza to, że klient jest oznaczany jako ryzykowny już wtedy, gdy model wyznaczy co najmniej 24,5% prawdopodobieństwa braku spłaty.
Tak niski próg zwiększa wykrywalność ryzyka, ale prowadzi również do większej liczby fałszywych alarmów.

# 14. Wybór najlepszego modelu

Najwyższy walidacyjny wynik F2 osiągnął: RandomForestClassifier

Model został następnie ponownie wytrenowany na połączonym zbiorze: 
     - treningowym + walidacyjnym
         
Zbiór testowy nadal pozostał nietknięty.

# 15. Końcowe wyniki na zbiorze testowym

Uzyskano:
- accuracy:           0,4697
- balanced accuracy:  0,6288
- precision:          0,2833
- recall:             0,9141
- F1:                 0,4326
- F2:                 0,6325
- ROC-AUC:            0,7864
- PR-AUC:             0,5602

Recall 91,4%
Model wykrył około 91% klientów, którzy nie spłacili zobowiązania.
    
Precision 28,3%
Spośród klientów oznaczonych jako ryzykowni około 28% rzeczywiście nie spłaciło zobowiązania.
Oznacza to dużą liczbę fałszywych alarmów.
    
Accuracy 47%
Niskie accuracy jest skutkiem bardzo ostrożnego progu 0,245. Model często wskazuje ryzyko, aby nie przeoczyć klienta, który może nie spłacić zadłużenia.
    
Nie oznacza to, że sam las źle rozróżnia klientów. Wskazuje na to ROC-AUC 0,786. To przyjęty próg powoduje dużą liczbę odrzuceń.
Bez znajomości rzeczywistych kosztów biznesowych nie można powiedzieć, że 0,245 jest najlepszym progiem wdrożeniowym.
    

# 16.Macierz Pomyłek

Macierz pomyłek przedstawia cztery możliwe przypadki:
- klient spłacił i model przewidział spłatę,
- klient spłacił, ale model przewidział brak spłaty,
- klient nie spłacił, ale model przewidział spłatę,
- klient nie spłacił i model prawidłowo wykrył ryzyko.
    
Najbardziej niebezpieczny dla banku jest zwykle:
 - klient nie spłacił, ale model przewidział spłatę

Jest to false negative.
    
Z kolei false positive oznacza odrzucenie klienta, który faktycznie spłaciłby zobowiązanie. To może powodować utratę dobrego klienta i przyszłych przychodów.

# 17. Interpretacja najważniejszych cech

Najważniejsze cechy modelu to:
- PAY_0 – najnowszy status spłaty,
- MAX_DELAY – największe opóźnienie,
- DELAY_MONTHS_COUNT – liczba miesięcy z opóźnieniem,
- PAY_2 – status spłaty w poprzednim miesiącu,
- UTILIZATION_MEAN – średnie wykorzystanie limitu.
    
Jest to logiczny wynik. Najwięcej informacji o przyszłym braku spłaty dostarcza dotychczasowa historia spłat.
Ważność cechy nie oznacza jednak związku przyczynowego. Informuje tylko, że model intensywnie wykorzystuje daną zmienną.

# 18. Zamiana prognozy na decyzję

Dla każdego klienta model zwraca prawdopodobieństwo: 
    - probability_of_default

Nastepnie stosowany jest próg:
    - predicted_default = probability_of_default >= 0.245

Rekomendacja została zapisana następująco:
- predicted_default = 1 → odmowa lub dodatkowa weryfikacja,
- predicted_default = 0 → można rozważyć przyznanie karty.
    
Celowo zastosowano określenie „można rozważyć”, ponieważ ostateczna decyzja powinna uwzględniać również regulamin banku, dochody, istniejące zobowiązania oraz ręczną analizę.

# 19. Zapis modelu

Do pliku zapisano nie tylko model, lecz także:
- próg decyzyjny,
- nazwy i kolejność cech,
- mapowanie błędnych kategorii,
- najlepsze hiperparametry,
- wyniki testowe.

kod 'joblib.dump(model_package, 'credit_default_model.joblib')'


mozna to wczytac w taki sposob:

    'model_package = joblib.load('credit_default_model.joblib')

model = model_package['model']
threshold = model_package['threshold']
feature_columns = model_package['feature_columns']'

Dzięki temu nie trzeba ponownie wykonywać GridSearchCV i trenować lasu.

# 20. Co należałoby zrobić przed wdrożeniem?

Przed wykorzystaniem modelu przez bank należałoby:
- ustalić rzeczywisty koszt niespłaconego kredytu,
- ustalić koszt odrzucenia dobrego klienta,
- ponownie dobrać próg według tych kosztów,
- sprawdzić kalibrację prawdopodobieństw,
- przeanalizować dyskryminację różnych grup klientów,
- sprawdzić zgodność z przepisami,
- monitorować skuteczność modelu w czasie,
- okresowo trenować model na nowszych danych.
    
Najważniejszy wniosek: model potrafi rozróżniać klientów ryzykownych od bezpieczniejszych, ale obecny próg został świadomie ustawiony bardzo ostrożnie. Przed prawdziwym wdrożeniem powinien zostać dopasowany do faktycznej polityki ryzyka banku.